# Imports

In [ ]:
# Imports
import re
import nltk
import numpy as np
import pandas as pd
from google.colab import drive

nltk.download('punkt', force=True)
nltk.download('punkt_tab', force=True)
nltk.download('popular', force=True)

from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize

import collections
from collections import defaultdict

from sklearn.model_selection import train_test_split
import pickle
import os

import math

import copy

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from google.colab import drive
import os

# Upload data

In [ ]:
# Get data
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/Datasets/NLP_v_0_datasets/test.csv"
df = pd.read_csv(file_path)

# Text Preprocessing

## Help Preprocessing Functions

In [ ]:
def transform_word_to_index(word, dictionary):
  """Transforms a word into an index using a dictionary"""
  try:
    return dictionary[word]
  except KeyError:
    return dictionary['<UNK>']

In [ ]:
def process_abbreviations(tokens):
  """Remove dots inside tokens if they look like letter sequences."""
  return [t.replace('.', '') if '.' in t and len(t) <= 5 else t for t in tokens]

In [ ]:
def get_length_list(sequences):
  length_list = []
  length = []

  for seqs in sequences:
    for seq in seqs:
      length.append(len(seq))

    # length.sort()
    length_list.append(length)
    length = []

  return length_list

In [ ]:
def max_sentences_len(text):
  return max(len(sent) for sent in text)

In [ ]:
def print_nested(data, indent=0):
    if isinstance(data, list):
        if any(isinstance(i, list) for i in data):
            print(' ' * indent + '[')
            for item in data:
                print_nested(item, indent + 4)
            print(' ' * indent + ']')
        else:
            print(' ' * indent + str(data))
    else:
        print(' ' * indent + str(data))

In [ ]:
def create_unified_word_index(texts, summaries, min_word_freq=1, max_vocab_size=None):
    """Creates a single dictionary for texts and summaries"""
    all_texts = list(texts) + list(summaries)
    return create_word_index(all_texts, min_word_freq=min_word_freq, max_vocab_size=max_vocab_size)

## Main Preprocessing Functions

In [ ]:
def normalize_abbreviations(text):
    """Finds sequences like U.S.A. or U.S. and returns them without dots (U S A -> USA).
    Keeps behavior simple: remove dots between single letters."""
    return re.sub(r'\b((?:[A-Za-z]\.){1,})([A-Za-z])?\b',
                  lambda m: m.group(0).replace('.', ''),
                  text)

In [ ]:
def clean_text_for_tokenization(text):
    """
    Clean text in a single place:
    - normalize abbreviations
    - keep sentence punctuation . ? !
    - remove other punctuation
    - lowercase and collapse spaces
    """
    text = normalize_abbreviations(str(text))
    # keep letters, digits, whitespace and . ? ! for sentence boundaries
    text = re.sub(r'[^A-Za-z0-9\s\.\?\!]', ' ', text)
    text = ' '.join(text.split()).lower()
    return text

In [ ]:
def create_word_index(texts, min_word_freq=1, max_vocab_size=None):
    """
    Create word2idx & idx2word with special tokens for multiple texts
    texts: iterable of strings
    """
    word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    idx2word = {v: k for k, v in word2idx.items()}

    word_counts = defaultdict(int)
    for text in texts:
      clear_text = clean_text_for_tokenization(text)
      tokens = word_tokenize(clear_text)
      tokens = process_abbreviations(tokens)
      for w in tokens:
        word_counts[w] += 1

    # Sort by freq desc then lexicographically for deterministic ordering
    sorted_words = sorted(word_counts.items(),
                          key=lambda x: (-x[1], x[0]))

    for word, count in sorted_words:
      if count < min_word_freq:
        continue
      if max_vocab_size and len(word2idx) >= max_vocab_size:
        break
      if word not in word2idx:
        idx = len(word2idx)
        word2idx[word] = idx
        idx2word[idx] = word

    return word2idx, idx2word

In [ ]:
def text_to_sequence(text, dictionary, max_len):
    """
    Convert raw text into a list of indices:
    [<SOS>, token..., <EOS>]
    Truncate if longer than max_len (including SOS/EOS).
    """
    clear_text = clean_text_for_tokenization(text)
    tokens = word_tokenize(clear_text)
    tokens = process_abbreviations(tokens)

    # Reserve 2 tokens for <SOS> and <EOS>
    max_tokens = max_len - 2
    if len(tokens) > max_tokens:
      tokens = tokens[:max_tokens]

    seq = [dictionary.get('<SOS>')]
    seq.extend([dictionary.get(t, dictionary.get('<UNK>')) for t in tokens])
    seq.append(dictionary.get('<EOS>'))

    return seq

In [ ]:
def texts_to_sequence(texts, dictionary, max_len):
  """Converts a list of texts using `text_to_sequence()`"""
  return [text_to_sequence(t, dictionary, max_len) for t in texts]

In [ ]:
def create_batches(sequences, pad_idx, batch_size=4):
  batches = []

  for i in range(0, len(sequences), batch_size):
    batch_sequences = sequences[i:i+batch_size]
    tensor_sequences = [torch.tensor(seq) for seq in batch_sequences]
    padded_batch = torch.nn.utils.rnn.pad_sequence(
        tensor_sequences,
        batch_first=True,
        padding_value=pad_idx
    )
    batches.append(padded_batch)

  return batches

  # for text in texts_sequences:
  #   for j in range(0, len(text), batch_size):
  #     # Separate
  #     batch = text[j:j+batch_size]

  #     # Add 0's
  #     # [0] - can be replaced with the <PAD> value from the dictionary (word_to_index['<PAD>'])
  #     padded_batch = [sent + [dictionary['<PAD>']]*(max_sentences_len(batch) - len(sent)) for sent in batch]

  #     # Batches for one text
  #     batches.append(padded_batch)

  #   # List of batched texts
  #   texts_bathes.append(batches)
  #   batches = []

  # return texts_bathes

In [ ]:
def replace_non_zero_numpy(data):
    arr = np.array(data)
    return np.where(arr != 0, 1, 0).tolist()

def generate_mask(texts_bathes):
  return [[replace_non_zero_numpy(inner) for inner in middle] for middle in texts_bathes]

# def generate_mask(texts_batches):
#   return [
#       [
#           [[1 if token != 0 else 0 for token in sent] for sent in batch]
#           for batch in text_batches
#       ]
#       for text_batches in texts_batches
#   ]

In [ ]:
def save_dictionaries(word2idx, idx2word, path):
    dictionaries = {'word2idx': word2idx, 'idx2word': idx2word}
    with open(path, 'wb') as f:
        pickle.dump(dictionaries, f)

In [ ]:
def open_loaded_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

In [ ]:
def train_val_test_split(X, y, test_size=0.15, val_ratio_of_train=0.1765):
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    # split X_temp into train and val
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_ratio_of_train, random_state=42)
    return X_train, X_val, X_test, y_train, y_val, y_test

## Dataset Functions

In [ ]:
class TextDataset(Dataset):
  def __init__(self, source_texts, target_texts, pad_idx=0):
    """
    Parameters:
      source_texts: list[list[int]] - articles in the form of index sequences
      target_texts: list[list[int]] - summaries in the form of index sequences
      pad_idx: - padding token index
    """
    self.source_texts = source_texts
    self.target_texts = target_texts
    self.pad_idx = pad_idx

    assert len(self.source_texts) == len(self.target_texts)

  def __len__(self):
    """Returns total number of samples"""
    return len(self.source_texts)

  def __getitem__(self, idx):
    """Returns one non-padding tensor by index"""
    return {
        'src': torch.tensor(self.source_texts[idx], dtype=torch.long),
        'tgt': torch.tensor(self.target_texts[idx], dtype=torch.long)
    }

In [ ]:
def collate_fn(batch, pad_idx):
  """
  Combines several samples into a batch
  Called automatically from DataLoader
  """
  src_sequences = [item['src'] for item in batch] # list[tensor]
  tgt_sequences = [item['tgt'] for item in batch] # list[tensor]

  src_padded = torch.nn.utils.rnn.pad_sequence(
      src_sequences,
      batch_first=True,
      padding_value=pad_idx
  )

  tgt_padded = torch.nn.utils.rnn.pad_sequence(
      tgt_sequences,
      batch_first=True,
      padding_value=pad_idx
  )

  return src_padded, tgt_padded

# Model Implementation

## Embedding Layer

In [ ]:
from torch import nn
import torch
import math

In [ ]:
class EmbeddingLayer(nn.Module):
  def __init__(self, vocab_size, pad_idx, embedding_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
    self.scale = math.sqrt(embedding_dim)

  def forward(self, x):
    embeddings = self.embedding(x)
    embeddings = torch.mul(embeddings, self.scale)
    return embeddings

In [ ]:
embedding_layer = EmbeddingLayer(
    vocab_size=1000,
    pad_idx=0,
    embedding_dim=4,
)

In [ ]:
embedding_layer

In [ ]:
inp = torch.LongTensor([[1,3,4,5,6], [5,5,5,5,5]])

In [ ]:
inp.shape

In [ ]:
output = embedding_layer(inp)

In [ ]:
output, output.shape

In [ ]:
output, output.shape

## Positional Encoding

In [ ]:
import numpy as np

import torch
from torch import nn
from torch import tensor

In [ ]:
def posenc(pos, d_model):
  div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
  # print(div_term)
  rep = np.zeros(d_model)
  rep[0::2] = np.sin(pos * div_term)
  rep[1::2] = np.cos(pos * div_term)
  return rep

In [ ]:
first = np.arange(0, 512, 2)
second = np.arange(0, 512, 2) * -(np.log(10000.0) / 512)
third = np.exp(np.arange(0, 512, 2) * -(np.log(10000.0) / 512))

In [ ]:
pos = posenc(3, 512)

In [ ]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_seq_len):
    super().__init__()

    pe = np.zeros((max_seq_len, d_model))
    positions = np.arange(max_seq_len)[:, None]

    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

    pe[:, 0::2] = np.sin(positions * div_term)
    pe[:, 1::2] = np.cos(positions * div_term)

    self.register_buffer("pe", torch.tensor(pe, dtype=torch.float32))

  def forward(self, x):
    seq_len = x.shape[1]
    return x + self.pe[:seq_len].unsqueeze(0)

In [ ]:
d_model = 2
positional_encoding = PositionalEncoding(d_model, max_seq_len=5000)
x = torch.zeros((1, 4, d_model)) # (batch, seq_len, d_model)
output = positional_encoding(x)
output

In [ ]:
output.shape

In [ ]:
arr = np.arange(5)[:, None]
arr.shape

## Causal Mask

In [ ]:
def create_causal_mask(seq_len, device, dtype):
  mask = torch.triu(
      torch.ones(seq_len, seq_len, device=device, dtype=dtype),
      diagonal=1
  )
  # mask = mask * float('-inf')
  mask = mask.masked_fill(mask == 1, float('-inf'))
  return mask.unsqueeze(0).unsqueeze(0) # (1, 1, L, L)

In [ ]:
mask = create_causal_mask(seq_len=4, device='cpu', dtype=torch.float32)

In [ ]:
print(mask)

In [ ]:
scores = torch.zeros(1, 1, 4, 4)
print(scores)
scores = scores + mask
print(scores)
probs = torch.softmax(scores, dim=-1)
print(probs)

In [ ]:
probs

## Padding Mask

In [ ]:
def create_padding_mask(src_tokens, pad_idx, device, dtype):
  mask = (src_tokens == pad_idx)
  mask = mask.float()
  mask = mask.masked_fill(mask == 1, float('-inf'))
  return mask.unsqueeze(1).unsqueeze(1).to(device=device, dtype=dtype)

In [ ]:
import torch

src_tokens = torch.tensor([
    [15, 28, 91, 2, 0, 0],
    [44, 18, 77, 31, 9, 0]
])

pad_idx = 0

In [ ]:
mask = create_padding_mask(
    src_tokens=src_tokens,
    pad_idx=pad_idx,
    device=src_tokens.device,
    dtype=torch.float32
)

In [ ]:
print(mask.shape)
print(mask)

## Combine Mask

In [ ]:
def combine_masks(padding_mask, causal_mask):
  return padding_mask + causal_mask

## Multi-Head Attention

In [ ]:
import torch
from torch import nn
from torch import tensor

import math

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(
      self,
      d_model: int,
      num_heads: int,
      dropout_rate: float,
      bias: bool
  ):
    super().__init__()

    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

    self.d_model = d_model
    self.num_heads = num_heads
    self.dropout_rate = dropout_rate
    self.bias = bias

    self.d_k = self.d_model // self.num_heads
    self.scale = 1 / math.sqrt(self.d_k)

    self.q_proj = nn.Linear(self.d_model, self.d_model, self.bias)
    self.k_proj = nn.Linear(self.d_model, self.d_model, self.bias)
    self.v_proj = nn.Linear(self.d_model, self.d_model, self.bias)
    self.out_proj = nn.Linear(self.d_model, self.d_model, self.bias)

    self.dropout_layer = nn.Dropout(self.dropout_rate)

  def forward(self, Q, K, V, mask=None):
    batch, Lq, _ = Q.shape # (batch, seq_len, d_model)
    _, Lk, _ = K.shape
    # -> ( batch, seq_len, d_model ) ->
    # -> ( batch, seq_len, num_heads, d_k ) ->
    # -> ( batch, num_heads, seq_len, d_k )
    Q_heads = self.q_proj(Q).reshape(batch, Lq, self.num_heads, self.d_k).transpose(1, 2)
    K_heads = self.k_proj(K).reshape(batch, Lk, self.num_heads, self.d_k).transpose(1, 2)
    V_heads = self.v_proj(V).reshape(batch, Lk, self.num_heads, self.d_k).transpose(1, 2)

    # attention scores
    scores = torch.matmul(Q_heads, K_heads.transpose(-2, -1))

    # scaling
    scores = scores * self.scale

    # masking
    mask = self.validate_mask(mask, scores)
    if mask is not None:
      scores = scores + mask

    # attention weights
    attn_weights = torch.softmax(scores, dim=-1)
    attn_weights = self.dropout_layer(attn_weights)

    # weighted sum of values
    attn_output = torch.matmul(attn_weights, V_heads).transpose(1,2).contiguous().reshape(batch, Lq, self.d_model)

    # output
    output = self.out_proj(attn_output)

    return output, attn_weights # optional return

  def validate_mask(self, mask, scores):
    if mask is None:
        return None

    # Check dimensions
    assert mask.dim() == 4, "Mask must be 4D (batch, heads/1, Lq, Lk)"

    B, H, Lq, Lk = scores.shape
    mB, mH, mLq, mLk = mask.shape

    # Shape compatibility
    assert mLk == Lk, "Mask last dim must match key length"
    assert mLq in (1, Lq), "Mask query dim must be 1 or Lq"
    assert mB in (1, B), "Mask batch dim must be 1 or batch size"
    assert mH in (1, H), "Mask head dim must be 1 or num_heads"

    # Dtype check
    assert mask.dtype.is_floating_point, "Mask must be float (additive mask)"

    # Device alignment
    mask = mask.to(device=scores.device, dtype=scores.dtype)

    return mask

## Feed Forward Network

In [ ]:
import torch
from torch import nn

In [ ]:
class FeedForward(nn.Module):
  def __init__(
    self,
    d_model,
    d_ff,
    bias: bool,
    activation_type = "gelu",
    dropout_rate = 0.1
    ):
    super().__init__()

    self.d_model = d_model
    self.dropout_rate = dropout_rate
    self.bias = bias
    self.activation = self._get_activation(activation_type)

    if d_ff is None:
      self.d_ff = 4 * d_model
    else:
      self.d_ff = d_ff

    self.fc1 = nn.Linear(self.d_model, self.d_ff, self.bias) # input projection
    self.fc2 = nn.Linear(self.d_ff, self.d_model, self.bias) # output projection

    self.dropout = nn.Dropout(self.dropout_rate)

  def forward(self, x):
    x = self.fc1(x) # (batch, seq_len, d_model) -> (batch, seq_len, d_ff)
    x = self.activation(x)
    x = self.dropout(x)
    x = self.fc2(x) # (batch, seq_len, d_ff) -> (batch, seq_len, d_model)
    x = self.dropout(x)
    return x

  def _get_activation(self, activation_type):
    """Helper method to return appropriate activation function"""
    activation_type = activation_type.lower()

    if activation_type == "relu":
      return nn.ReLU()
    elif activation_type == "gelu":
      return nn.GELU()
    else:
      raise ValueError(f"Unsupported activation type: '{activation_type}'. "
                f"Supported types: 'relu', 'gelu'")

## Add & Layer Normalization

In [ ]:
import torch
from torch import nn

In [ ]:
class AddNorm(nn.Module):
  def __init__(
    self,
    d_model,
    dropout_rate,
    elementwise_affine: bool,
    eps=1e-5
    ):
    super().__init__()

    self.d_model = d_model
    self.dropout_rate = dropout_rate
    self.eps = eps

    self.layer_norm = nn.LayerNorm(self.d_model, self.eps, elementwise_affine)
    self.dropout = nn.Dropout(self.dropout_rate)

  def forward(self, x, sublayer_output):
    return self.layer_norm(x + self.dropout(sublayer_output))

## Encoder Layer

In [ ]:
import torch
from torch import nn

In [ ]:
class EncoderLayer(nn.Module):
  def __init__(
    self,
    d_model,
    num_heads,
    d_ff,
    dropout_rate,
    bias,
    activation_type,
    eps=1e-5,
    elementwise_affine=True
    ):
    super().__init__()

    self.d_model = d_model
    self.num_heads = num_heads
    self.d_ff = d_ff
    self.dropout_rate = dropout_rate
    self.eps = eps
    self.bias = bias
    self.activation_type = activation_type

    self.attention = MultiHeadAttention(
      d_model=self.d_model,
      num_heads=self.num_heads,
      dropout_rate=self.dropout_rate,
      bias=self.bias
    )
    self.feed_forward = FeedForward(
      d_model=self.d_model,
      d_ff=self.d_ff,
      bias=self.bias,
      activation_type=self.activation_type,
      dropout_rate=self.dropout_rate
    )
    self.add_norm_1 = AddNorm(
      d_model=self.d_model,
      dropout_rate=self.dropout_rate,
      elementwise_affine=elementwise_affine,
      eps=self.eps
    )
    self.add_norm_2 = AddNorm(
      d_model=self.d_model,
      dropout_rate=self.dropout_rate,
      elementwise_affine=elementwise_affine,
      eps=self.eps
    )

  def forward(self, x, src_mask):
    attn_out, _ = self.attention(x, x, x, mask=src_mask)
    x = self.add_norm_1(x, attn_out)
    ffn_out = self.feed_forward(x)
    x = self.add_norm_2(x, ffn_out)
    return x

## Encoder Stack

In [ ]:
import torch
from torch import nn

In [ ]:
class EncoderStack(nn.Module):
  def __init__(
    self,
    d_model,
    num_heads,
    d_ff,
    dropout_rate,
    bias,
    activation_type,
    elementwise_affine,
    max_seq_len,
    num_layers,
    eps=1e-5,
    ):
    super().__init__()

    self.d_model = d_model
    self.dropout_rate = dropout_rate
    self.eps = eps

    self.positional_encoding = PositionalEncoding(
      self.d_model,
      max_seq_len
    )

    self.dropout = nn.Dropout(self.dropout_rate)

    self.encoder_layers = nn.ModuleList([
      EncoderLayer(
        d_model=self.d_model,
        num_heads=num_heads,
        d_ff=d_ff,
        dropout_rate=self.dropout_rate,
        bias=bias,
        activation_type=activation_type,
        eps=self.eps,
        elementwise_affine=elementwise_affine
        ) for i in range(num_layers)
    ])

    self.layer_norm = nn.LayerNorm(
      self.d_model,
      self.eps,
      elementwise_affine
    )

  def forward(self, src_embeddings, src_mask):
    x = self.positional_encoding(src_embeddings)
    x = self.dropout(x)
    for layer in self.encoder_layers:
      x = layer(x, src_mask)
    x = self.layer_norm(x)
    return x

## Decoder Layer

In [ ]:
import torch
from torch import nn

In [ ]:
class DecoderLayer(nn.Module):
  def __init__(
    self,
    d_model,
    num_heads,
    d_ff,
    dropout_rate,
    bias,
    activation_type,
    eps=1e-5,
    elementwise_affine=True
    ):
    super().__init__()

    self.d_model = d_model
    self.dropout_rate = dropout_rate
    self.eps = eps

    self.self_attention = MultiHeadAttention(
      self.d_model,
      num_heads,
      self.dropout_rate,
      bias
    )

    self.cross_attention = MultiHeadAttention(
      self.d_model,
      num_heads,
      self.dropout_rate,
      bias
    )

    self.feed_forward = FeedForward(
      self.d_model,
      d_ff,
      bias,
      activation_type,
      self.dropout_rate
    )

    self.norm1 = AddNorm(
      self.d_model,
      self.dropout_rate,
      elementwise_affine,
      self.eps
    )

    self.norm2 = AddNorm(
      self.d_model,
      self.dropout_rate,
      elementwise_affine,
      self.eps
    )

    self.norm3 = AddNorm(
      self.d_model,
      self.dropout_rate,
      elementwise_affine,
      self.eps
    )

  def forward(self, y, encoder_output, src_mask, tgt_mask):
    self_attn_out, _ = self.self_attention(y, y, y, mask=tgt_mask)
    y = self.norm1(y, self_attn_out)

    cross_attn_out, _ = self.cross_attention(
      y,
      encoder_output,
      encoder_output,
      mask=src_mask
    )
    y = self.norm2(y, cross_attn_out)

    ffn_out = self.feed_forward(y)
    y = self.norm3(y, ffn_out)

    return y

## Decoder Stack

In [ ]:
class DecoderStack(nn.Module):
  def __init__(
    self,
    d_model,
    num_heads,
    d_ff,
    dropout_rate,
    bias,
    activation_type,
    max_seq_len,
    num_layers,
    elementwise_affine=True,
    eps=1e-5,
    ):
    super().__init__()

    self.d_model = d_model
    self.dropout_rate = dropout_rate
    self.eps = eps

    self.positional_encoding = PositionalEncoding(
      self.d_model,
      max_seq_len
    )

    self.dropout = nn.Dropout(self.dropout_rate)

    self.decoder_layers = nn.ModuleList([
      DecoderLayer(
          d_model=self.d_model,
          num_heads=num_heads,
          d_ff=d_ff,
          dropout_rate=self.dropout_rate,
          bias=bias,
          activation_type=activation_type,
          eps=self.eps,
          elementwise_affine=elementwise_affine
      ) for i in range(num_layers)
    ])

    self.layer_norm = nn.LayerNorm(
      self.d_model,
      self.eps,
      elementwise_affine
    )

  def forward(self, tgt_embeddings, encoder_output, src_mask, tgt_mask):
    x = self.positional_encoding(tgt_embeddings)
    x = self.dropout(x)
    for layer in self.decoder_layers:
      x = layer(x, encoder_output, src_mask, tgt_mask)
    x = self.layer_norm(x)
    return x

## Output Projection Head

In [ ]:
import torch
from torch import nn

In [ ]:
class OutputProjectionHead(nn.Module):
  def __init__(
    self,
    d_model,
    vocab_size,
    bias
    ):
    super().__init__()

    self.d_model = d_model
    self.vocab_size = vocab_size

    self.output_projection = nn.Linear(self.d_model, self.vocab_size, bias=bias)

  def forward(self, decoder_output):
    logits = self.output_projection(decoder_output)
    return logits

## Transformer Wrapper

In [ ]:
import torch
from torch import nn

In [ ]:
class TransformerSummarizer(nn.Module):
  def __init__(
    self,
    src_vocab_size,
    tgt_vocab_size,
    src_pad_idx,
    tgt_pad_idx,
    d_model,
    num_heads,
    d_ff,
    num_encoder_layers,
    num_decoder_layers,
    dropout_rate,
    max_seq_len,
    bias,
    activation_type,
    elementwise_affine,
    eps=1e-5
    ):
    super().__init__()

    self.d_model = d_model
    self.src_pad_idx = src_pad_idx
    self.tgt_pad_idx = tgt_pad_idx

    self.src_embedding = EmbeddingLayer(
      src_vocab_size,
      self.src_pad_idx,
      self.d_model
    )

    self.tgt_embedding = EmbeddingLayer(
      tgt_vocab_size,
      self.tgt_pad_idx,
      self.d_model
    )

    self.encoder = EncoderStack(
      self.d_model,
      num_heads,
      d_ff,
      dropout_rate,
      bias,
      activation_type,
      elementwise_affine,
      max_seq_len,
      num_encoder_layers,
      eps
    )

    self.decoder = DecoderStack(
      self.d_model,
      num_heads,
      d_ff,
      dropout_rate,
      bias,
      activation_type,
      max_seq_len,
      num_decoder_layers,
      elementwise_affine,
      eps
    )

    self.output_head = OutputProjectionHead(
      self.d_model,
      tgt_vocab_size,
      bias
    )

  def forward(self, src_tokens, tgt_tokens):
    device = src_tokens.device
    dtype = next(self.parameters()).dtype

    src_mask = create_padding_mask(src_tokens, self.src_pad_idx, device, dtype)
    tgt_mask = create_padding_mask(tgt_tokens, self.tgt_pad_idx, device, dtype)
    causal_mask = create_causal_mask(tgt_tokens.shape[1], device, dtype)
    combine_mask = combine_masks(tgt_mask, causal_mask)

    src_embeddings = self.src_embedding(src_tokens)

    encoder_output = self.encoder(
      src_embeddings,
      src_mask
    )

    tgt_embeddings = self.tgt_embedding(tgt_tokens)

    decoder_output = self.decoder(
      tgt_embeddings,
      encoder_output,
      src_mask,
      combine_mask
    )

    logits = self.output_head(decoder_output)

    return logits

# Main Pipeline

### Text Preprocessing

In [ ]:
# Upload dataset

file_path = "/content/drive/MyDrive/Datasets/NLP_v_0_datasets/test.csv"

if not os.path.exists(file_path):
  raise FileNotFoundError(f"Dataset not found at {file_path}")

df = pd.read_csv(file_path)
texts = df['article'].astype(str).tolist()
summaries = df['highlights'].astype(str).tolist()

In [ ]:
min_word_freq = 2
max_vocab_size = 10000
word_2_idx, idx_2_word = create_unified_word_index(texts, summaries, min_word_freq=min_word_freq, max_vocab_size=max_vocab_size)
print(f"Vocabulary size: {len(word_2_idx)}")

In [ ]:
# Convert to sequences
max_len = 64
X = texts_to_sequence(texts, word_2_idx, max_len)
y = texts_to_sequence(summaries, word_2_idx, max_len)

# Split
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y)

In [ ]:
print(X_train)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

pad_idx = word_2_idx['<PAD>']

In [ ]:
# Datasets & loaders
train_dataset = TextDataset(X_train, y_train, pad_idx=pad_idx)
val_dataset = TextDataset(X_val, y_val, pad_idx=pad_idx)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          collate_fn=lambda batch: collate_fn(batch, pad_idx),
                          num_workers=2, pin_memory=(device.type == 'cuda'))

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=lambda batch: collate_fn(batch, pad_idx),
                        num_workers=2, pin_memory=(device.type == 'cuda'))